# Model routing with a decision model

The companion to `tool_selection_decision.ipynb`. There, a decision model picks
which **tool** an agent should use. Here, it picks which **model** should serve
the request. Same idea, same decision model — a different thing to choose.

Why a decision model and not a chat model as the classifier? Choosing from a
fixed set is a bounded question with one right answer. A chat model answers it by
generating text, so it can drift or name an option that isn't on the list. The
**OpenAI Decisions API** (`gpt-6-luna`) returns one of the candidates you gave it,
with a confidence and no generated text — the choice can't be hallucinated.

Strands' `ModelRouter` owns the per-request model choice; *which* model is a
pluggable `RoutingStrategy`. The decision model classifies the request against
each candidate's **description** (it knows nothing about the models themselves —
only the names and descriptions you pass). Mirrors `model_routing_with_jev.py`;
only the strategy differs (Jev → OpenAI). Per the Strands docs, the router is
passed as the agent's model, not via plugins.

**Setup:** AWS creds for the candidate Bedrock models; `OPENAI_API_KEY` and
`pip install "openai>=3.26.0"`.

In [ ]:
import os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("AWS_REGION", "us-east-1")
from dotenv import load_dotenv
load_dotenv()

## The routing strategy

One bounded `choice` over the router's candidates, via the async OpenAI client.
It records each decision (`chosen_model`, `confidence`) so the routing is
visible. Routes the opening attempt only; declines (`None`) after a failed
serving attempt so the router surfaces the error rather than cycling.

In [ ]:
from typing import Any
from openai import AsyncOpenAI
from strands.models import RoutingCandidate, RoutingContext, RoutingStrategy

class OpenAIDecisionsRoutingStrategy(RoutingStrategy):
    def __init__(self) -> None:
        self._oai = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.decisions = []   # (chosen_model_name, confidence) per routed request

    async def select(self, context: RoutingContext, **kwargs: Any) -> RoutingCandidate | None:
        if context.attempts:
            return None
        query = ""
        for m in reversed(context.messages):
            if m.get("role") == "user":
                query = "".join(b.get("text", "") for b in m.get("content", []) if "text" in b)
                break
        choices = [{"value": c.name, "description": c.description or c.name} for c in context.candidates]
        resp = await self._oai.decisions.create(
            model="gpt-6-luna", input=query,
            questions=[{"type": "choice", "name": "model",
                        "instructions": "Which model should handle this request?", "choices": choices}],
        )
        answer = next(a for a in resp.answers if getattr(a, "name", None) == "model")
        self.decisions.append((answer.choice, answer.confidence))
        return next((c for c in context.candidates if c.name == answer.choice), None)

## Route an agent

Two candidates, described by the kind of work each suits: a small model for
routine lookups, a strong one for multi-step reasoning. The decision model reads
those descriptions and routes each request. The agent invocation API is
unchanged.

In [ ]:
from strands import Agent
from strands.models import BedrockModel, ModelRouter, RoutingCandidate

strategy = OpenAIDecisionsRoutingStrategy()
router = ModelRouter(
    models=[
        RoutingCandidate(
            BedrockModel(model_id="us.amazon.nova-lite-v1:0", region_name=os.environ["AWS_REGION"]),
            name="routine",
            description="Short factual questions and routine lookups.",
        ),
        RoutingCandidate(
            BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0", region_name=os.environ["AWS_REGION"]),
            name="advanced",
            description="Multi-step reasoning, systems design, trade-off analysis.",
        ),
    ],
    strategy=strategy,
)
agent = Agent(model=router, callback_handler=None)

QUESTIONS = [
    "What time zone is Tokyo in?",
    "What is the capital of Australia?",
    "Design a rollback-safe migration from regional to global idempotency keys.",
    "Compare optimistic vs pessimistic locking for a high-contention ledger and recommend one.",
]

for q in QUESTIONS:
    n = len(strategy.decisions)
    r = await agent.invoke_async(q)
    chosen, conf = strategy.decisions[n]
    print(f"Q: {q}")
    print(f"   routed to: {chosen}  (confidence {conf})")
    print(f"   answer:    {str(r)[:90]}\n")

## The decisions, on their own

Every routing choice the Decisions API made. The routine lookups go to the small
model, the reasoning tasks to the strong one — a bounded choice each time, never
a generated guess.

In [ ]:
for q, (chosen, conf) in zip(QUESTIONS, strategy.decisions):
    print(f"{chosen:9s} (confidence {conf})  <-  {q}")